# How I Built an AI Agent That Reads the News For Me with CrewAI

url: https://medium.com/ai-in-plain-english/build-your-first-daily-information-digest-agent-with-crewai-5a2e5670a981

A step-by-step guide to building an AI agent that filters tech news and delivers only what matters — using free tools and APIs.

##  Here’s what we’ll build:

— An AI agent that searches the web for tech/AI news

— Automatic filtering based on your interests

— Summaries with source links

In [2]:
# Environment

import os
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["OLLAMA_API_KEY"]=os.getenv("OLLAMA_API_KEY")
import google.genai as genai


In [4]:
# Use a CrewAI-compatible LLM wrapper for local Ollama models
from crewai import LLM
from ollama import Client, ChatResponse


client = Client(
    host = "http://ollama.com",
    headers = {"Authorization": f"Bearer {os.getenv('OLLAMA_API_KEY')}"},
)
AGENT_LLM = LLM(
    model="openai/gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0.2,
)


In [8]:
# Defining tools with crewai-tools

from langchain_community.tools import DuckDuckGoSearchRun
from crewai.tools import tool

search_tool_instance = DuckDuckGoSearchRun()

@tool("DuckDuckGo Search")
def search_tool(query: str) -> str:
    """A tool for searching the web using DuckDuckGo."""
    return search_tool_instance.run(query)

In [9]:
# Defining Agents

from crewai import Agent


collector_agent = Agent(
    role="Content Collector",
    goal="Gather the latest updates and articles specifically about IT, AI advancements, AI agents, and the tech market from trusted public sources for daily review.",
    backstory="""
    You are a highly focused content collector who specializes in tracking the most relevant updates in technology and AI.
    Your task is to find daily news, insights, and breakthroughs that matter to professionals and enthusiasts,
    ignoring unrelated or low-value content. You prioritize relevance, timeliness, and authority of sources.""",
    verbose=False,
    llm=AGENT_LLM,
    tools=[search_tool],
    max_iter=2,
 )

##  Creating Tasks
We need to create a Task for the agent. A task is essentially a work assignment that includes a Description, which provides clear instructions on what the agent should do; an Expected Output, which defines what the final deliverable should look like.

In [13]:
from crewai import Task
# this task is assigned to the collector_agent, which will use the search_tool to gather information and updates based on the specified description and expected output.

collector_task = Task(
    description="""
    Collect the latest public updates and articles related to:
    1. IT industry developments
    2. AI advancements and research
    3. AI agents and frameworks
    4. Tech market news and trends

    Focus on content from the past 24–48 hours. Gather information that is relevant, high-quality,
    and actionable for professionals who want to stay informed daily.
    """,
    expected_output="A curated list of recent articles, news items, and updates in IT, AI, AI agents, and the tech market, with key points summarized for each item.",
    agent=collector_agent
)


##  Executing the Crew
Combine the agent and task into a Crew, execute it, and review the output.

In [14]:
from crewai import Crew, Process

collector_crew = Crew(
    agents= [collector_agent],
    tasks=[collector_task],
    process=Process.sequential,
    verbose=True
)

print("@ Launching Collector Crew...")
research_result = collector_crew.kickoff()
from IPython.display import Markdown

print("Collector REPORT COMPLETE")
Markdown (research_result.raw)

@ Launching Collector Crew...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1b25c4cc-6363-4f80-9e93-4d5ec79f2f23                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Collect the latest public updates and articles related to:                                                 │
│      1. IT industry developments                                                                                │
│      2. AI advancements and research                                                                            │
│      3. AI agents and frameworks                                                                                │
│      4. Tech market news and trends                                                                             │
│                                                                                                                 │
│      Focus on content from the past 24–48 hours. Gather information that is relevant, high-quality,             │
│      and actionable for professionals who want to stay informed daily.                                          │
│                                                                                                                 │
│  ID: 9552e8db-0c52-460f-a654-6a1f42168b09                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Collector                                                                                       │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Collect the latest public updates and articles related to:                                                 │
│      1. IT industry developments                                                                                │
│      2. AI advancements and research                                                                            │
│      3. AI agents and frameworks                                                                                │
│      4. Tech market news and trends                                                                             │
│                                                                                                                 │
│      Focus on content from the past 24–48 hours. Gather information that is relevant, high-quality,             │
│      and actionable for professionals who want to stay informed daily.                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#5) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: duck_duck_go_search                                                                                      │
│  Args: {'query': 'latest IT industry developments October 2023'}                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Collect the latest public updates and articles related to:                                                 │
│      1. IT industry developments                                                                                │
│      2. AI advancements and research                                                                            │
│      3. AI agents and frameworks                                                                                │
│      4. Tech market news and trends                                                                             │
│                                                                                                                 │
│      Focus on content from the past 24–48 hours. Gather information that is relevant, high-quality,             │
│      and actionable for professionals who want to stay informed daily.                                          │
│                                                                                                                 │
│  Agent: Content Collector                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 1b25c4cc-6363-4f80-9e93-4d5ec79f2f23                                                                       │
│  Final Output: **1. IT Industry Developments**                                                                  │
│                                                                                                                 │
│  - **Title:** Google Previews Mobile OS Updates Ahead of Developer Conference                                   │
│    - **URL:** [Reuters](https://www.reuters.com)                                                                │
│    - **Publication Date:** 19 hours ago                                                                         │
│    - **Description:** Google has provided a sneak peek into upcoming updates for its mobile operating system    │
│  ahead of its developer conference, highlighting new features and improvements.                                 │
│                                                                                                                 │
│  **2. AI Advancements and Research**                                                                            │
│                                                                                                                 │
│  - **Title:** Google Research and DeepMind's 2023 AI Progress                                                   │
│    - **URL:** [Reuters](https://www.reuters.com)                                                                │
│    - **Publication Date:** October 2023                                                                         │
│    - **Description:** Google Research and DeepMind have made significant strides in AI technology throughout    │
│  2023, with notable advancements in generative AI, including the launch of Bard for text generation and         │
│  translation in over 40 languages.                                                                              │
│                                                                                                                 │
│  - **Title:** Generative AI's Impact on NLP and LLM                                                             │
│    - **URL:** [Reuters](https://www.reuters.com)                                                                │
│    - **Publication Date:** October 2023                                                                         │
│    - **Description:** Generative AI has revolutionized Natural Language Processing and Large Language Models,   │
│  impacting various sectors such as industry, education, and healthcare, while also presenting new risks.        │
│                                                                                                                 │
│  **3. AI Agents and Frameworks**                                                                                │
│                                                                                                                 │
│  - **Title:** Anthropic's AI Agents for Complex Tasks                                                           │
│    - **URL:** [Jeff Su                                                                                          │
│  Academy](https://academy.jeffsu.org/ai-toolkit?utm_source=youtube&utm_medium=video&utm_campaign=177)           │
│    - **Publication Date:** October 2023                                                                         │
│    - **Description:** Anthropic has announced AI agent

Collector REPORT COMPLETE


**1. IT Industry Developments**

- **Title:** Google Previews Mobile OS Updates Ahead of Developer Conference
  - **URL:** [Reuters](https://www.reuters.com)
  - **Publication Date:** 19 hours ago
  - **Description:** Google has provided a sneak peek into upcoming updates for its mobile operating system ahead of its developer conference, highlighting new features and improvements.

**2. AI Advancements and Research**

- **Title:** Google Research and DeepMind's 2023 AI Progress
  - **URL:** [Reuters](https://www.reuters.com)
  - **Publication Date:** October 2023
  - **Description:** Google Research and DeepMind have made significant strides in AI technology throughout 2023, with notable advancements in generative AI, including the launch of Bard for text generation and translation in over 40 languages.

- **Title:** Generative AI's Impact on NLP and LLM
  - **URL:** [Reuters](https://www.reuters.com)
  - **Publication Date:** October 2023
  - **Description:** Generative AI has revolutionized Natural Language Processing and Large Language Models, impacting various sectors such as industry, education, and healthcare, while also presenting new risks.

**3. AI Agents and Frameworks**

- **Title:** Anthropic's AI Agents for Complex Tasks
  - **URL:** [Jeff Su Academy](https://academy.jeffsu.org/ai-toolkit?utm_source=youtube&utm_medium=video&utm_campaign=177)
  - **Publication Date:** October 2023
  - **Description:** Anthropic has announced AI agents designed for complex tasks, competing with OpenAI. These systems are evolving beyond chatbots, posing new risks as they develop rapidly.

- **Title:** LangChain: Open Source Framework for AI Agents
  - **URL:** [Jeff Su Academy](https://academy.jeffsu.org/ai-toolkit?utm_source=youtube&utm_medium=video&utm_campaign=177)
  - **Publication Date:** October 2023
  - **Description:** LangChain offers an open-source framework with pre-built agent architecture and integrations, allowing for adaptable AI agent development as the ecosystem evolves.

**4. Tech Market News and Trends**

- **Title:** Global Technology News and Trends
  - **URL:** [Reuters](https://www.reuters.com)
  - **Publication Date:** 1 day ago
  - **Description:** Stay updated with the latest technology news globally, covering top companies like Google and Apple, as well as emerging startups, with insights into hardware, apps, and more.

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [12]:
collector_agent = Agent(
    role="Content Collector",
    goal="Gather latest updates and articles specifically about IT, AI advancements, AI agents, and the tech market from trusted public sources for daily review.",
    backstory="""
    You are a highly focused content collector who specializes in tracking the most relevant updates in technology and AI.
    For every article or update, you must include the title, full URL, publication date, and a short description (1-2 sentences).
    Ignore irrelevant content.
    """,
    verbose=False,
    llm=AGENT_LLM,
    tools=[search_tool],
    max_iterations=2
    )